# Hierarchical (Multilevel) Models

Hierarchical models allow parameters to vary by group while sharing information across groups.
This notebook covers:
1. **No pooling vs complete pooling vs partial pooling**
2. **Shrinkage** effect
3. **Hierarchical model** specification
4. **Implementation** from scratch with MCMC

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

%matplotlib inline
np.random.seed(42)
print('Setup complete.')

## 1. Motivating Example: Batting Averages

Suppose we observe the batting average for several players, each with a different
number of at-bats. Players with few at-bats have noisy estimates.

**Question:** How do we estimate each player's true ability?

In [ ]:
# Simulate data: 8 players with different sample sizes
true_mu = 0.27  # population mean batting average
true_tau = 0.03  # between-player std

n_players = 8
true_thetas = np.random.normal(true_mu, true_tau, n_players)
true_thetas = np.clip(true_thetas, 0.15, 0.40)

# Different sample sizes (at-bats)
at_bats = np.array([10, 15, 20, 50, 100, 200, 300, 500])
hits = np.array([np.random.binomial(n, theta) for n, theta in zip(at_bats, true_thetas)])
observed_avg = hits / at_bats

print('Player | At-bats | Hits | Observed avg | True theta')
print('-' * 55)
for i in range(n_players):
    print(f'  {i+1:4d}  | {at_bats[i]:7d} | {hits[i]:4d} | {observed_avg[i]:12.3f} | {true_thetas[i]:.3f}')

## 2. Three Approaches

| Approach | Estimate | Problem |
|----------|----------|---------|
| **No pooling** | $\hat\theta_j = k_j / n_j$ | Noisy for small $n_j$ |
| **Complete pooling** | $\hat\theta = \sum k_j / \sum n_j$ | Ignores individual variation |
| **Partial pooling** | Hierarchical model | Best of both worlds |

In [ ]:
# No pooling: raw observed average
no_pool = observed_avg.copy()

# Complete pooling: overall average
complete_pool = np.full(n_players, hits.sum() / at_bats.sum())

# Partial pooling: empirical Bayes shrinkage (James-Stein style)
# Posterior mean under Normal-Normal model:
# theta_j_post = (n_j * y_j / sigma^2 + mu / tau^2) / (n_j / sigma^2 + 1 / tau^2)
# We use moment estimates for mu and tau
overall_mean = hits.sum() / at_bats.sum()
sigma2_approx = overall_mean * (1 - overall_mean)  # binomial variance approx
between_var = max(np.var(observed_avg) - np.mean(sigma2_approx / at_bats), 0.0001)

partial_pool = np.zeros(n_players)
for j in range(n_players):
    precision_data = at_bats[j] / sigma2_approx
    precision_prior = 1.0 / between_var
    partial_pool[j] = (precision_data * observed_avg[j] + precision_prior * overall_mean) / (precision_data + precision_prior)

print('Estimates:')
print(f'{"Player":>7s} {"No Pool":>8s} {"Full Pool":>10s} {"Partial":>8s} {"Truth":>7s}')
for j in range(n_players):
    print(f'{j+1:7d} {no_pool[j]:8.3f} {complete_pool[j]:10.3f} {partial_pool[j]:8.3f} {true_thetas[j]:7.3f}')

In [ ]:
# Visualise shrinkage
fig, ax = plt.subplots(figsize=(10, 6))

for j in range(n_players):
    ax.annotate('', xy=(partial_pool[j], j), xytext=(no_pool[j], j),
                arrowprops=dict(arrowstyle='->', color='steelblue', lw=1.5))

ax.scatter(no_pool, range(n_players), color='red', s=100, zorder=5, label='No pooling (MLE)')
ax.scatter(partial_pool, range(n_players), color='steelblue', s=100, zorder=5, label='Partial pooling')
ax.scatter(true_thetas, range(n_players), marker='x', color='black', s=100, zorder=5, label='True value')
ax.axvline(overall_mean, ls='--', color='gray', label=f'Grand mean = {overall_mean:.3f}')

ax.set_yticks(range(n_players))
ax.set_yticklabels([f'Player {j+1} (n={at_bats[j]})' for j in range(n_players)])
ax.set_xlabel('Batting Average')
ax.set_title('Shrinkage: No Pooling -> Partial Pooling')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

print('Note: Players with fewer at-bats are shrunk more towards the grand mean.')

## 3. Hierarchical Model Structure

$$\theta_j \sim \mathcal{N}(\mu, \tau^2) \quad \text{(group-level prior)}$$
$$y_j \mid \theta_j \sim \text{Binomial}(n_j, \theta_j) \quad \text{(likelihood)}$$
$$\mu \sim \mathcal{N}(0.25, 0.1^2), \quad \tau \sim \text{HalfNormal}(0.1)$$

The key insight: groups **share information** through the hyperparameters $\mu$ and $\tau$.

In [ ]:
# MCMC for the hierarchical model (Gibbs-like with MH steps)
def log_joint(thetas, mu, log_tau):
    tau = np.exp(log_tau)
    # Hyperpriors
    lp = stats.norm.logpdf(mu, 0.25, 0.1)
    lp += stats.halfnorm.logpdf(tau, scale=0.1) + log_tau  # Jacobian
    # Group-level priors
    for j in range(n_players):
        if thetas[j] <= 0 or thetas[j] >= 1:
            return -np.inf
        lp += stats.norm.logpdf(thetas[j], mu, tau)
        lp += hits[j] * np.log(thetas[j]) + (at_bats[j] - hits[j]) * np.log(1 - thetas[j])
    return lp

# Run MCMC
n_iter = 30000
burn = 10000
dim = n_players + 2  # thetas + mu + log_tau
chain = np.zeros((n_iter, dim))
current = np.concatenate([observed_avg, [overall_mean, np.log(0.03)]])
current_lp = log_joint(current[:n_players], current[-2], current[-1])
prop_std = np.concatenate([np.full(n_players, 0.02), [0.01, 0.1]])
acc = 0

for i in range(n_iter):
    proposal = current + np.random.normal(0, prop_std)
    proposal_lp = log_joint(proposal[:n_players], proposal[-2], proposal[-1])
    if np.log(np.random.rand()) < proposal_lp - current_lp:
        current = proposal
        current_lp = proposal_lp
        acc += 1
    chain[i] = current

chain = chain[burn:]
print(f'Acceptance rate: {acc/n_iter:.3f}')
print(f'Posterior mean of mu: {chain[:, -2].mean():.4f}')
print(f'Posterior mean of tau: {np.exp(chain[:, -1]).mean():.4f}')

In [ ]:
# Compare estimates
hier_means = chain[:, :n_players].mean(axis=0)

# MSE comparison
mse_no_pool = np.mean((no_pool - true_thetas)**2)
mse_full_pool = np.mean((complete_pool - true_thetas)**2)
mse_partial = np.mean((partial_pool - true_thetas)**2)
mse_hier = np.mean((hier_means - true_thetas)**2)

print('Mean Squared Error comparison:')
print(f'  No pooling:       {mse_no_pool:.6f}')
print(f'  Complete pooling:  {mse_full_pool:.6f}')
print(f'  Partial pooling:   {mse_partial:.6f}')
print(f'  Hierarchical MCMC: {mse_hier:.6f}')

## Key Takeaways

- **No pooling** ignores shared structure; **complete pooling** ignores group differences.
- **Partial pooling** (hierarchical models) provides the best of both worlds.
- **Shrinkage** is adaptive: groups with less data are pulled more towards the grand mean.
- Hierarchical models are ubiquitous: clinical trials, education, sports, A/B testing.
- In practice, use **PyMC** or **Stan** for efficient sampling of hierarchical models.